# Surface phytoplankton and zooplankton

I compare surface $P$ and $Z$ across the two runs.

Use the Julia kernel and run the cells in order. Keep this notebook in `/nfs/roberts/pi/pi_ey239/qi/figures/ice_iron`.

- $P$ and $Z$: $\mathrm{mmol\,N\,m^{-3}}$
- Chlorophyll: $\mathrm{mg\,Chl\,m^{-3}}$


In [1]:
using Pkg

model_root = "/nfs/roberts/pi/pi_ey239/qi/global-ocean-model"

isfile(joinpath(model_root, "Project.toml")) ||
    error("Project.toml not found: $model_root")

Pkg.activate(model_root)

println("Project: ", Base.active_project())
Pkg.status(["OceanBioME", "Oceananigans"])

  Activating 

Project: /nfs/roberts/pi/pi_ey239/qi/global-ocean-model/Project.toml


project at `/nfs/roberts/pi/pi_ey239/qi/global-ocean-model`


Status `/nfs/roberts/pi/pi_ey239/qi/global-ocean-model/Project.toml`
⌃ [a49af516] OceanBioME v0.18.0
⌅ [9e8cae18] Oceananigans v0.110.14
Info Packages marked with ⌃ and ⌅ have new versions available. Those with ⌃ may be upgradable, but those with ⌅ are restricted by compatibility constraints from upgrading. To see why use `status --outdated`


In [2]:
using OceanBioME, Markdown
using Oceananigans: CPU, RectilinearGrid

parameter_grid = RectilinearGrid(CPU(); size=(2, 2, 2), extent=(2, 2, 2))
parameter_bgc = LOBSTER(parameter_grid;
    limiting_nutrients=(:nitrate, :ammonia, :iron))
plankton = parameter_bgc.underlying_biogeochemistry.plankton

println("LOBSTER defaults | OceanBioME ", Base.pkgversion(OceanBioME))

rows = ["| parameter | value | unit |", "|:--|--:|:--|"]
for (name, unit) in (
    (:maximum_grazing_rate, raw"$\mathrm{s^{-1}}$"),
    (:grazing_half_saturation, raw"$\mathrm{mmol\,N\,m^{-3}}$"),
    (:preference_for_phytoplankton, "—"),
    (:phytoplankton_maximum_growth_rate, raw"$\mathrm{s^{-1}}$"),
    (:phytoplankton_mortality_rate, raw"$\mathrm{m^3\,(mmol\,N)^{-1}\,s^{-1}}$"),
    (:zooplankton_mortality_rate, raw"$\mathrm{m^3\,(mmol\,N)^{-1}\,s^{-1}}$"),
    (:zooplankton_excretion_rate, raw"$\mathrm{s^{-1}}$"),
    (:zooplankton_assimilation_fraction, "—"),
    (:chlorophyll_ratio, raw"$\mathrm{mg\,Chl\,(mmol\,N)^{-1}}$"),
)
    push!(rows, "| `$(name)` | $(getproperty(plankton, name)) | $(unit) |")
end
display(MIME"text/markdown"(), Markdown.parse(join(rows, "\n")))


LOBSTER defaults | OceanBioME 0.18.0


| parameter                           |   value | unit                                   |
|:----------------------------------- | -------:|:-------------------------------------- |
| `maximum_grazing_rate`              | 9.26e-6 | $\mathrm{s^{-1}}$                      |
| `grazing_half_saturation`           |     1.0 | $\mathrm{mmol\,N\,m^{-3}}$             |
| `preference_for_phytoplankton`      |     0.5 | —                                      |
| `phytoplankton_maximum_growth_rate` | 2.42e-5 | $\mathrm{s^{-1}}$                      |
| `phytoplankton_mortality_rate`      |  5.8e-7 | $\mathrm{m^3\,(mmol\,N)^{-1}\,s^{-1}}$ |
| `zooplankton_mortality_rate`        | 2.31e-6 | $\mathrm{m^3\,(mmol\,N)^{-1}\,s^{-1}}$ |
| `zooplankton_excretion_rate`        |  5.8e-7 | $\mathrm{s^{-1}}$                      |
| `zooplankton_assimilation_fraction` |     0.7 | —                                      |
| `chlorophyll_ratio`                 |    1.31 | $\mathrm{mg\,Chl\,(mmol\,N)^{-1}}$     |


In [3]:
using Dates

# Settings
notebook_root = "/nfs/roberts/pi/pi_ey239/qi/figures/ice_iron"
run_names = ["ice_glorys01_2_iron", "ice_glorys01_2_iron_2"] # Oldest first.
comparison_name = last(run_names) * "_merged"
day0 = DateTime(2000, 1, 1) # Model day 0.

segment_files = [[joinpath(model_root, "output_ice", "surface_fields_$(name)_rank$(r).jld2")
                  for r in 0:1] for name in run_names]
figure_dir = joinpath(notebook_root, "zooplankton_" * comparison_name)
first_day = 0.0
last_day = nothing        # nothing = all available times.
map_months = nothing      # nothing = all complete months.
months_per_page = 4
ratio_floor = 1e-8        # Minimum P for the ratio.
map_spacing_deg = 1.0     # Map spacing only.
map_radius_deg = 1.5

regions = [
    (name="Global", south=-90.0, north=90.0),
    (name="Southern Ocean (<40°S)", south=-90.0, north=-40.0),
    (name="Tropics (20°S-20°N)", south=-20.0, north=20.0),
    (name="Northern Ocean (>40°N)", south=40.0, north=90.0),
]

# Match these to the run settings.
# Set to nothing to skip grazing estimates.
grazing_parameters = (
    preference=0.5,
    maximum_rate=9.26e-6,   # s^-1
    half_saturation=1.0,   # mmol N m^-3
    assimilation=0.7,
    excretion_rate=5.8e-7, # s^-1
    mortality_rate=2.31e-6, # m^3 (mmol N)^-1 s^-1
)

using JLD2, Oceananigans, CairoMakie, Statistics, Printf
CairoMakie.activate!()
println("Julia ", VERSION, " | project: ", Base.active_project())
for dep in values(Pkg.dependencies())
    dep.name in ("Oceananigans", "OceanBioME") && println(dep.name, " ", dep.version)
end
println("Grazing parameters: ", grazing_parameters)


[ Info: Precompiling CairoMakie [13f3f980-e62b-5c42-98c6-ff1f3baf88f0] (cache misses: target mismatch (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task no

Julia 1.12


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


.4


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


 | project: /nfs/roberts/pi/pi_ey239/qi/global-ocean-model/Project.toml


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up



OceanBioME 0.18.0
Oceananigans 0.110.14
Grazing parameters: (preference = 0.5, maximum_rate = 9.26e-6, half_saturation = 1.0, assimilation = 0.7, excretion_rate = 5.8e-7, mortality_rate = 2.31e-6)


## Read the model output

For overlapping times, I keep the later record with both ranks available. Each time is counted once; land and halo cells are excluded.


In [4]:
module ChlComparison

using Dates, Statistics, Printf, JLD2, Oceananigans
using Oceananigans.Architectures: CPU, on_architecture
using Oceananigans.Grids: inactive_cell

export model_catalog, monthly_means, nearest_mapping, remap,
       coordinate_edges, cell_weights, compare_fields, print_month_plan,
       read_record, write_catalog

const SECONDS_PER_DAY = 86400.0

function geometry(file, variable)
    p = "timeseries/$variable/serialized/grid_index"
    gp = haskey(file, p) ? "serialized/grid_$(file[p])" : "serialized/grid"
    haskey(file, gp) || error("Missing saved grid for $variable: $gp")
    grid = on_architecture(CPU(), file[gp])
    g = hasproperty(grid, :underlying_grid) ? grid.underlying_grid : grid
    all(n -> hasproperty(g, n), (:Nx, :Ny, :Nz, :λᶜᶜᵃ, :φᶜᶜᵃ)) ||
        error("Unsupported saved grid: $(typeof(g)); no fabricated coordinates are used.")
    nx, ny, nz = Int(g.Nx), Int(g.Ny), Int(g.Nz)
    lon = vec(Float64.(Array(g.λᶜᶜᵃ[1:nx, 1:ny])))
    lat = vec(Float64.(Array(g.φᶜᶜᵃ[1:nx, 1:ny])))
    all(isfinite, lon) && all(y -> isfinite(y) && -90 <= y <= 90, lat) ||
        error("Invalid saved center coordinates")
    location = file["timeseries/$variable/serialized/location"]
    all(l -> l === Center || l isa Center, location) ||
        error("Expected a center-located surface tracer, got $location")
    wet = vec([!inactive_cell(i, j, nz, grid) for i in 1:nx, j in 1:ny])
    return (; nx, ny, nz, lon=mod.(lon .+ 180, 360) .- 180, lat, wet)
end

function surface_values(file, variable, key, g)
    raw = Array(file["timeseries/$variable/$key"])
    if ndims(raw) == 3 && size(raw, 3) == 1
        raw = dropdims(raw; dims=3)
    end
    ndims(raw) == 2 || error("$variable must contain one saved surface layer; got $(size(raw))")
    dx, dy = size(raw, 1)-g.nx, size(raw, 2)-g.ny
    dx >= 0 && dy >= 0 && iseven(dx) && iseven(dy) || error("Incompatible saved halos")
    hx, hy = dx÷2, dy÷2
    inds = file["timeseries/$variable/serialized/indices"]
    for (ind, n, h) in zip(inds[1:2], (g.nx, g.ny), (hx, hy))
        ind isa Colon && continue
        ind isa AbstractUnitRange && first(ind) == 1-h && last(ind) == n+h ||
            error("Partial horizontal slices are not supported: $inds")
    end
    zind = inds[3]
    (zind == g.nz || (zind isa AbstractUnitRange && first(zind) == last(zind) == g.nz)) ||
        error("Saved $variable is not explicitly the top model layer: indices=$inds, Nz=$(g.nz)")
    return vec(Float64.(raw[hx+1:hx+g.nx, hy+1:hy+g.ny]))
end

function time_table(file)
    rows = [(time=Float64(file["timeseries/t/$k"]), key=String(k))
            for k in keys(file["timeseries/t"]) if tryparse(Int, String(k)) !== nothing]
    isempty(rows) && error("No saved model times")
    all(r -> isfinite(r.time) && r.time >= 0, rows) || error("Invalid model time")
    sort!(rows; by=r -> r.time)
    all(diff([r.time for r in rows]) .> 0) || error("Duplicate model times inside one file")
    return rows
end

function with_model_files(f, paths)
    files = Any[]
    try
        for p in paths
            push!(files, jldopen(p, "r"))
        end
        return f(files)
    finally
        foreach(close, files)
    end
end

same_geometry(a, b) = (a.nx, a.ny, a.nz) == (b.nx, b.ny, b.nz) &&
                       a.lon == b.lon && a.lat == b.lat && a.wet == b.wet

# Match rank pairs, then keep the later complete record at overlapping times.
function model_catalog(input_paths; day0=DateTime(2000, 1, 1), tolerance=1.0,
                       variables=["chl"], segment_names=nothing,
                       first_day=0.0, last_day=nothing)
    isempty(input_paths) && error("No model files supplied")
    segments = all(p -> p isa AbstractString, input_paths) ? [collect(input_paths)] : collect.(input_paths)
    all(s -> length(s) == 2, segments) || error("Each segment needs exactly rank0 and rank1")
    paths = abspath.(String.(vcat(segments...)))
    length(unique(paths)) == length(paths) || error("Supply distinct ocean rank files, without repeating segments")
    all(isfile, paths) || error("Missing model file(s): $(join(filter(p -> !isfile(p), paths), ", "))")
    0 <= tolerance < 60 || error("Use a nonnegative time tolerance smaller than 60 seconds")
    first_day >= 0 || error("first_day must be nonnegative")
    isnothing(last_day) || last_day >= first_day || error("last_day precedes first_day")
    variables = unique(String.(collect(variables)))
    isempty(variables) && error("At least one required variable is needed")
    names = isnothing(segment_names) ? ["segment_$i" for i in eachindex(segments)] : String.(segment_names)
    length(names) == length(segments) && length(unique(names)) == length(names) || error("Segment names must be distinct and match file pairs")
    in_window(t) = t >= first_day*SECONDS_PER_DAY-tolerance &&
                   (isnothing(last_day) || t <= last_day*SECONDS_PER_DAY+tolerance)
    return with_model_files(paths) do files
        for (path, file) in zip(paths, files), variable in variables
            haskey(file, "timeseries/$variable") || error("Missing saved $variable in $path")
        end
        geoms = [geometry(files[i], first(variables)) for i in 1:2]
        for (index, file) in enumerate(files), variable in variables
            geo = geometry(file, variable)
            same_geometry(geo, geoms[mod1(index, 2)]) ||
                error("Grid/location/wet mask differs between segments or variables: $(paths[index]), $variable. Do not merge different grids.")
        end
        tables = time_table.(files)
        candidates, rejected, duplicates = NamedTuple[], String[], NamedTuple[]
        for segment in eachindex(segments)
            file_ids = (2segment-1, 2segment)
            table0, table1 = tables[file_ids[1]], tables[file_ids[2]]
            # Keep the original model times across restarts.
            used_second = Set{String}()
            accepted = 0
            for r0 in table0
                in_window(r0.time) || continue
                matches = findall(r -> abs(r.time-r0.time) <= tolerance, table1)
                length(matches) <= 1 || error("Ambiguous rank times in $(names[segment]) near day $(r0.time/SECONDS_PER_DAY)")
                if isempty(matches)
                    push!(rejected, "$(names[segment]) day $(r0.time/SECONDS_PER_DAY): no matching rank1 time")
                    continue
                end
                r1 = table1[only(matches)]
                r1.key in used_second && error("A rank1 record would be reused in $(names[segment])")
                rows = (r0, r1)
                if !all(haskey(files[file_ids[k]], "timeseries/$v/$(rows[k].key)") for k in 1:2, v in variables)
                    push!(rejected, "$(names[segment]) day $(r0.time/SECONDS_PER_DAY): incomplete required fields on one rank")
                    continue
                end
                # Stop if a stored array is incomplete or has the wrong shape.
                for k in 1:2, v in variables
                    surface_values(files[file_ids[k]], v, rows[k].key, geoms[k])
                end
                push!(used_second, r1.key)
                push!(candidates, (time=r0.time, times=(r0.time, r1.time), keys=(r0.key, r1.key),
                                   files=file_ids, segment, segment_name=names[segment],
                                   datetime=day0+Millisecond(round(Int, 1000*r0.time))))
                accepted += 1
            end
            for r1 in table1
                in_window(r1.time) && !(r1.key in used_second) &&
                    push!(rejected, "$(names[segment]) rank1 day $(r1.time/SECONDS_PER_DAY): no accepted paired record")
            end
            println(names[segment], ": ", accepted, " complete paired snapshots before deduplication")
        end
        isempty(candidates) && error("No complete paired snapshots in requested interval")
        sort!(candidates; by=r -> r.time)
        records = NamedTuple[]
        first_index = 1
        while first_index <= length(candidates)
            last_index = first_index
            anchor = candidates[first_index].time
            while last_index < length(candidates) && candidates[last_index+1].time-anchor <= tolerance
                last_index += 1
            end
            group = candidates[first_index:last_index]
            length(unique(r.segment for r in group)) == length(group) ||
                error("Two snapshots in the same segment fall within time tolerance near day $(anchor/SECONDS_PER_DAY)")
            winner = group[argmax(getproperty.(group, :segment))]
            push!(records, winner)
            for loser in group
                loser.segment == winner.segment && continue
                push!(duplicates, (time=winner.time, datetime=winner.datetime,
                                   kept=winner.segment_name, discarded=loser.segment_name,
                                   kept_keys=winner.keys, discarded_keys=loser.keys))
            end
            first_index = last_index+1
        end
        # Keep each group within the timestamp tolerance.
        all(diff(getproperty.(records, :time)) .> tolerance) ||
            error("Ambiguous timestamp cluster; inspect saved times or reduce tolerance")
        println("Model source files (read-only):\n", join(paths, '\n'))
        println("Overlapping records removed: ", length(duplicates), "; later complete segment has priority")
        println("Retained snapshots: ", length(records), "; model day ", first(records).time/SECONDS_PER_DAY,
                "–", last(records).time/SECONDS_PER_DAY)
        println("Calendar coverage: ", first(records).datetime, " → ", last(records).datetime)
        for segment in eachindex(names)
            println("  Retained from ", names[segment], ": ", count(r -> r.segment == segment, records))
        end
        !isempty(rejected) && println("Excluded incomplete/unmatched records: ", length(rejected), "; first: ", first(rejected))
        return (; paths, segments=[paths[2i-1:2i] for i in eachindex(segments)], segment_names=names,
                 day0, tolerance, variables, geoms, records, rejected, duplicates,
                 lon=vcat([g.lon for g in geoms]...), lat=vcat([g.lat for g in geoms]...),
                 wet=vcat([g.wet for g in geoms]...))
    end
end

function read_record(files, catalog, record, variable)
    indices = hasproperty(record, :files) ? record.files : (1, 2)
    return vcat([surface_values(files[indices[k]], variable, record.keys[k], catalog.geoms[k]) for k in 1:2]...)
end

# Record which files supplied each time.
function write_catalog(directory, catalog)
    mkpath(directory)
    quote_csv(x) = "\"" * replace(string(x), "\""=>"\"\"") * "\""
    open(joinpath(directory, "merged_snapshot_sources.csv"), "w") do io
        println(io, "model_day,datetime,segment,rank0_file,rank0_key,rank1_file,rank1_key")
        for r in catalog.records
            row = (r.time/SECONDS_PER_DAY, r.datetime, r.segment_name,
                   catalog.paths[r.files[1]], r.keys[1], catalog.paths[r.files[2]], r.keys[2])
            println(io, join(quote_csv.(row), ','))
        end
    end
    open(joinpath(directory, "merge_report.txt"), "w") do io
        println(io, "Priority, oldest to newest: ", join(catalog.segment_names, " -> "))
        println(io, "Required variables: ", join(catalog.variables, ", "))
        println(io, "Timestamp tolerance (seconds): ", catalog.tolerance)
        println(io, "Duplicate records removed: ", length(catalog.duplicates))
        for d in catalog.duplicates
            println(io, d.datetime, ": kept ", d.kept, "; discarded ", d.discarded)
        end
        println(io, "\nRejected records: ", length(catalog.rejected))
        foreach(r -> println(io, r), catalog.rejected)
    end
    return directory
end

function month_plan(catalog)
    first_month = Date(year(first(catalog.records).datetime), month(first(catalog.records).datetime), 1)
    last_month = Date(year(last(catalog.records).datetime), month(last(catalog.records).datetime), 1)
    plans = NamedTuple[]
    for m in first_month:Month(1):last_month
        expected = collect(DateTime(m):Day(1):DateTime(m+Month(1)))
        indices = Int[]
        missing_dates = DateTime[]
        for d in expected
            t = Dates.value(d-catalog.day0)/1000
            match = findall(r -> all(x -> abs(x-t) <= catalog.tolerance, r.times), catalog.records)
            length(match) > 1 && error("Ambiguous model snapshot near $d")
            isempty(match) ? push!(missing_dates, d) : push!(indices, only(match))
        end
        push!(plans, (; month=m, ndays=daysinmonth(m), indices, missing_dates, complete=isempty(missing_dates)))
    end
    return plans
end

function print_month_plan(catalog)
    plans = month_plan(catalog)
    for p in plans
        if p.complete
            println(p.month, ": complete; ", p.ndays, " days, ", length(p.indices), " instantaneous endpoints")
        else
            println(p.month, ": SKIP incomplete month; missing ", length(p.missing_dates),
                    " endpoints (first missing: ", first(p.missing_dates), ")")
        end
    end
    return plans
end

# Monthly means from daily snapshots.
function monthly_means(catalog; plans=month_plan(catalog))
    results = Dict{Date,Vector{Float64}}()
    return with_model_files(catalog.paths) do files
        for p in plans
            p.complete || continue
            accum = zeros(length(catalog.wet))
            for (n, idx) in enumerate(p.indices)
                r = catalog.records[idx]
                values = read_record(files, catalog, r, "chl")
                bad = catalog.wet .& .!isfinite.(values)
                any(bad) && error("Nonfinite model chl in wet cells at $(r.datetime): count=$(count(bad)); month not averaged")
                weight = (n == 1 || n == length(p.indices)) ? 0.5/p.ndays : 1.0/p.ndays
                accum[catalog.wet] .+= weight .* values[catalog.wet]
            end
            accum[.!catalog.wet] .= NaN
            results[p.month] = accum
            println("Model monthly mean ", p.month, ": wet min/max=", extrema(accum[catalog.wet]),
                    "; nonpositive wet cells=", count(x -> x <= 0, accum[catalog.wet]))
        end
        isempty(results) && error("No complete calendar month is available. Keep partial months out of a full-month comparison.")
        return results
    end
end

# Nearest model cell; apply the land mask after mapping.
function nearest_mapping(lon, lat, targetlon, targetlat; radius=1.5)
    0 < radius <= 5 || error("Use a positive, explicitly chosen radius ≤ 5 degrees")
    xyz = (cosd.(lat).*cosd.(lon), cosd.(lat).*sind.(lon), sind.(lat))
    mapping = zeros(Int, length(targetlon), length(targetlat))
    distances = fill(NaN, size(mapping))
    Threads.@threads for j in eachindex(targetlat)
        y = targetlat[j]
        candidates = findall(v -> abs(v-y) <= radius, lat)
        for i in eachindex(targetlon)
            x = targetlon[i]
            tx, ty, tz = cosd(y)*cosd(x), cosd(y)*sind(x), sind(y)
            best, index = -Inf, 0
            for k in candidates
                dot = tx*xyz[1][k]+ty*xyz[2][k]+tz*xyz[3][k]
                if dot > best
                    best, index = dot, k
                end
            end
            distance = acosd(clamp(best, -1, 1))
            if index > 0 && distance <= radius
                mapping[i, j], distances[i, j] = index, distance
            end
        end
    end
    return (; mapping, distances)
end

function remap(values, mapping, wet)
    result = fill(NaN, size(mapping))
    for i in eachindex(mapping)
        k = mapping[i]
        if k > 0 && wet[k]
            result[i] = values[k]
        end
    end
    return result
end

function coordinate_edges(centers; latitude=false)
    length(centers) >= 2 && all(diff(centers) .> 0) || error("Coordinates must be increasing")
    edges = vcat(centers[1]-(centers[2]-centers[1])/2,
                 (centers[1:end-1] .+ centers[2:end])./2,
                 centers[end]+(centers[end]-centers[end-1])/2)
    latitude && (edges = clamp.(edges, -90.0, 90.0))
    return edges
end

# Cell weights for a regular longitude-latitude grid.
function cell_weights(lon, lat)
    le = coordinate_edges(lon)
    pe = coordinate_edges(lat; latitude=true)
    all(diff(le) .> 0) && all(diff(pe) .> 0) || error("Invalid inferred cell bounds")
    return deg2rad.(diff(le)) * transpose(diff(sind.(pe)))
end

function weighted_correlation(x, y, w)
    mx, my = sum(w.*x), sum(w.*y)
    dx, dy = x.-mx, y.-my
    vx, vy = sum(w.*dx.^2), sum(w.*dy.^2)
    return vx > 0 && vy > 0 ? sum(w.*dx.*dy)/sqrt(vx*vy) : NaN
end

function compare_fields(model, obs, weights; month)
    size(model) == size(obs) == size(weights) || error("Comparison grids disagree")
    valid = isfinite.(model) .& isfinite.(obs) .& (obs .> 0) .& (weights .> 0)
    any(valid) || error("No common valid ocean pixels in $month")
    x, y, w = model[valid], obs[valid], weights[valid]
    w ./= sum(w)
    mean_model, mean_sat = sum(w.*x), sum(w.*y)
    difference = x.-y
    positive = valid .& (model .> 0)
    log_rmse, log_bias, r_log10, geometric_ratio = NaN, NaN, NaN, NaN
    if any(positive)
        lx, ly, lw = log10.(model[positive]), log10.(obs[positive]), weights[positive]
        lw ./= sum(lw)
        log_bias = sum(lw.*(lx.-ly))
        log_rmse = sqrt(sum(lw.*(lx.-ly).^2))
        geometric_ratio = 10.0^log_bias
        r_log10 = weighted_correlation(lx, ly, lw)
    end
    obs_valid = isfinite.(obs) .& (obs .> 0)
    summary = (; month, n=count(valid), n_log=count(positive), n_model_nonpositive=count(valid .& (model .<= 0)),
               observation_area_covered=sum(weights[valid])/sum(weights[obs_valid]),
               mean_model, mean_sat, bias=sum(w.*difference), rmse=sqrt(sum(w.*difference.^2)),
               r_linear=weighted_correlation(x, y, w), log10_bias=log_bias, log10_rmse=log_rmse,
               r_log10, geometric_model_sat_ratio=geometric_ratio)
    return (; valid, positive, summary)
end

end


Main.ChlComparison

In [5]:
module ZooDiagnostics

using Dates, Statistics, Printf, JLD2, Oceananigans
using Oceananigans.Architectures: CPU, on_architecture
using ..ChlComparison: model_catalog, read_record, with_model_files,
                       month_plan, nearest_mapping

export inspect_run, diagnose, display_mapping, write_summary, grazing_rates

const DAY = 86400.0

# Cell areas from the model grid.
function native_area(file, variable, geo)
    index_path = "timeseries/$variable/serialized/grid_index"
    grid_path = haskey(file, index_path) ? "serialized/grid_$(file[index_path])" : "serialized/grid"
    grid = on_architecture(CPU(), file[grid_path])
    g = hasproperty(grid, :underlying_grid) ? grid.underlying_grid : grid
    hasproperty(g, :Azᶜᶜᵃ) || error("Saved grid has no native horizontal cell-area metric")
    area = vec(Float64.(Array(g.Azᶜᶜᵃ[1:geo.nx, 1:geo.ny])))
    all(x -> isfinite(x) && x > 0, area[geo.wet]) || error("Invalid wet-cell areas")
    return area
end

# Read complete rank pairs across the runs.
function inspect_run(paths; day0=DateTime(2000, 1, 1), first_day=0.0,
                     last_day=nothing, grazing=true, tolerance=1.0, segment_names=nothing)
    variables = grazing ? ["P", "Z", "chl", "sPOM"] : ["P", "Z", "chl"]
    catalog = model_catalog(paths; day0, first_day, last_day, tolerance,
                            variables, segment_names)
    area = with_model_files(catalog.paths) do files
        vcat([native_area(files[i], "P", catalog.geoms[i]) for i in 1:2]...)
    end
    gaps = diff(getproperty.(catalog.records, :time))./DAY
    any(gaps .> 1+tolerance/DAY) &&
        println("NOTE: gaps between snapshots exist; monthly averages require every daily endpoint.")
    return merge(catalog, (; variables, area))
end

# Snapshot grazing rates in mmol N m^-3 day^-1 (OceanBioME 0.18.0).
function grazing_rates(P, Z, D, pars)
    names = (:gp, :gtot, :assimilation, :z_excretion, :z_mortality, :z_bio_tendency)
    all(isfinite, (P, Z, D)) && min(P, Z, D) >= 0 ||
        return NamedTuple{names}(ntuple(_ -> NaN, 6))
    w, K, g = pars.preference, pars.half_saturation, pars.maximum_rate
    denominator = w*P + (1-w)*D
    preference = denominator > 0 ? w*P/denominator : 0.0
    food = preference*P + (1-preference)*D
    saturation = food^2/(food^2 + K^2)
    gtot = DAY*g*saturation*Z
    gp = food > 0 ? gtot*preference*P/food : 0.0
    assimilation = pars.assimilation*gtot
    z_excretion = DAY*pars.excretion_rate*Z
    z_mortality = DAY*pars.mortality_rate*Z^2
    return (; gp, gtot, assimilation, z_excretion, z_mortality,
             z_bio_tendency=assimilation-z_excretion-z_mortality)
end

function validate_parameters(p)
    isnothing(p) && return
    all(isfinite, values(p)) || error("Grazing parameters must be finite")
    0 <= p.preference <= 1 && 0 <= p.assimilation <= 1 || error("Invalid fraction")
    p.maximum_rate >= 0 && p.half_saturation > 0 &&
        p.excretion_rate >= 0 && p.mortality_rate >= 0 || error("Invalid rate parameter")
end

weighted_mean(x, area, mask) = any(mask) ? sum(area[mask].*x[mask])/sum(area[mask]) : NaN

function diagnose(catalog; regions, months=nothing,
                  parameters=nothing, ratio_floor=1e-8)
    validate_parameters(parameters)
    ratio_floor > 0 || error("ratio_floor must be positive")
    isnothing(parameters) || "sPOM" in catalog.variables || error("Inspect with grazing=true first")
    length(unique(r.name for r in regions)) == length(regions) || error("Region names must be distinct")
    any(r.name == "Global" for r in regions) || error("Include a region named Global")
    all(-90 <= r.south < r.north <= 90 for r in regions) || error("Invalid latitude bounds")
    masks = [catalog.wet .& (catalog.lat .>= r.south) .&
             (r.north == 90 ? catalog.lat .<= r.north : catalog.lat .< r.north) for r in regions]
    plans = month_plan(catalog)
    isnothing(months) || (plans = filter(p -> p.month in months, plans))
    for p in plans
        println(p.month, p.complete ? ": complete monthly map" : ": skipped (missing daily endpoints)")
    end
    complete = filter(p -> p.complete, plans)
    monthly = [(month=p.month, P=zeros(length(catalog.wet)), Z=zeros(length(catalog.wet)),
                chl=zeros(length(catalog.wet)), gp=zeros(length(catalog.wet))) for p in complete]
    contributions = Dict{Int,Vector{Tuple{Int,Float64}}}()
    for (mi, p) in enumerate(complete), (j, idx) in enumerate(p.indices)
        weight = (j == 1 || j == length(p.indices)) ? 0.5/p.ndays : 1.0/p.ndays
        push!(get!(contributions, idx, Tuple{Int,Float64}[]), (mi, weight))
    end
    series = NamedTuple[]
    initial = nothing
    with_model_files(catalog.paths) do files
        for (idx, record) in enumerate(catalog.records)
            fields = Dict(v => read_record(files, catalog, record, v) for v in catalog.variables)
            P, Z, chl = fields["P"], fields["Z"], fields["chl"]
            for v in catalog.variables
                bad = catalog.wet .& .!isfinite.(fields[v])
                any(bad) && error("Nonfinite $v in $(count(bad)) wet cells at $(record.datetime). Inspect data before interpreting growth.")
            end
            n = length(P)
            rate_names = (:gp, :gtot, :assimilation, :z_excretion, :z_mortality, :z_bio_tendency)
            rates = NamedTuple{rate_names}(ntuple(_ -> fill(NaN, n), 6))
            if !isnothing(parameters)
                for k in findall(catalog.wet)
                    value = grazing_rates(P[k], Z[k], fields["sPOM"][k], parameters)
                    for name in rate_names
                        getproperty(rates, name)[k] = getproperty(value, name)
                    end
                end
            end
            if idx == 1
                valid_ratio = catalog.wet .& (P .> ratio_floor) .& (chl .>= 0)
                ratios = chl[valid_ratio]./P[valid_ratio]
                initial = (; day=record.time/DAY, P=weighted_mean(P, catalog.area, catalog.wet),
                           Z=weighted_mean(Z, catalog.area, catalog.wet),
                           chl_per_P=isempty(ratios) ? NaN : median(ratios),
                           chl_per_P_range=isempty(ratios) ? (NaN, NaN) : extrema(ratios))
                println("First saved state: day ", initial.day, "; mean P=", initial.P,
                        "; mean Z=", initial.Z, " mmol N m^-3; Z/P=", initial.Z/initial.P)
                println("Saved chl/P: median=", initial.chl_per_P, "; min/max=", initial.chl_per_P_range,
                        " mg Chl/mmol N (expected 1.31 for the inspected default)")
            end
            for (region, mask) in zip(regions, masks)
                p, z = weighted_mean(P, catalog.area, mask), weighted_mean(Z, catalog.area, mask)
                rmask = mask .& isfinite.(rates.gp)
                rmeans = NamedTuple{rate_names}(Tuple(weighted_mean(getproperty(rates, name), catalog.area, rmask)
                                                     for name in rate_names))
                paired_p = weighted_mean(P, catalog.area, rmask)
                total_area = sum(catalog.area[mask])
                push!(series, merge((; day=record.time/DAY, datetime=record.datetime, region=region.name,
                                      source_segment=record.segment, source_run=catalog.segment_names[record.segment],
                                      P=p, Z=z, chl=weighted_mean(chl, catalog.area, mask),
                                      ratio=p > ratio_floor && z >= 0 ? z/p : NaN), rmeans,
                                     (; grazing_pressure=paired_p > ratio_floor ? rmeans.gp/paired_p : NaN,
                                        valid_fraction=total_area > 0 ? sum(catalog.area[rmask])/total_area : NaN,
                                        negative_P=count(mask .& (P .< 0)), negative_Z=count(mask .& (Z .< 0)))))
            end
            for (mi, weight) in get(contributions, idx, Tuple{Int,Float64}[])
                for name in (:P, :Z, :chl)
                    getproperty(monthly[mi], name) .+= weight .* fields[String(name)]
                end
                monthly[mi].gp .+= weight .* rates.gp
            end
            (idx == 1 || idx % 30 == 0 || idx == length(catalog.records)) &&
                println("Processed ", idx, "/", length(catalog.records), " snapshots (day ", record.time/DAY, ")")
        end
    end
    for m in monthly, name in (:P, :Z, :chl, :gp)
        getproperty(m, name)[.!catalog.wet] .= NaN
    end
    neg = sum(r.negative_P+r.negative_Z for r in series if r.region == "Global")
    neg > 0 && println("NOTE: ", neg, " negative P/Z cell-records. Linear means retain them; log plots omit them.")
    return (; catalog, series, monthly, parameters, initial, ratio_floor)
end

function display_mapping(catalog; spacing=1.0, radius=1.5)
    0.25 <= spacing <= 5 || error("Choose map display spacing between 0.25 and 5 degrees")
    lon = collect(-180+spacing/2:spacing:180-spacing/2)
    lat = collect(-90+spacing/2:spacing:90-spacing/2)
    match = nearest_mapping(catalog.lon, catalog.lat, lon, lat; radius)
    return (; map=match.mapping, lon, lat)
end

function write_summary(path, diagnostics)
    rows = diagnostics.series
    isempty(rows) && error("No summaries")
    encode(x) = x isa AbstractString ? "\"" * replace(x, "\""=>"\"\"") * "\"" : string(x)
    open(path, "w") do io
        println(io, join(string.(keys(first(rows))), ','))
        for r in rows
            println(io, join(encode.(values(r)), ','))
        end
    end
    return path
end

end


Main.ZooDiagnostics

In [6]:
module ZooPlots

using CairoMakie, Dates, Printf

export time_series, grazing_series, monthly_maps

const COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
                "#E69F00", "#56B4E9", "#6F4C9B", "#333333"]

positive(values) = [isfinite(v) && v > 0 ? Float64(v) : NaN for v in values]
finite(values) = [isfinite(v) ? Float64(v) : NaN for v in values]

function global_rows(diagnostics)
    rows = sort([r for r in diagnostics.series if lowercase(r.region) == "global"]; by=r -> r.day)
    isempty(rows) && error("No Global time-series rows are available to plot.")
    return rows
end

function inside_legend(ax; position=:lt)
    axislegend(ax; position, labelsize=13, rowgap=4, patchsize=(22, 12),
               backgroundcolor=(:white, 0.93))
end

function no_positive_message!(ax, values)
    if !any(v -> isfinite(v) && v > 0, values)
        ylims!(ax, 1e-3, 1.0)
        text!(ax, 0.5, 0.45; text="No positive values", space=:relative,
              align=(:center, :center), fontsize=15)
    end
end

function decade_ticks!(ax, values)
    good = filter(v -> isfinite(v) && v > 0, values)
    isempty(good) && return
    lo, hi = extrema(log10.(good))
    ticks = 10.0 .^ collect(floor(Int, lo):ceil(Int, hi))
    ax.yticks = ticks
end

function normalized(values)
    first_positive = findfirst(v -> isfinite(v) && v > 0, values)
    isnothing(first_positive) && return fill(NaN, length(values))
    return positive(values) ./ values[first_positive]
end

# Surface means and changes relative to the starting values.
function time_series(diagnostics)
    rows = global_rows(diagnostics)
    regions = unique([r.region for r in diagnostics.series])
    x = [r.day for r in rows]
    p, z = [r.P for r in rows], [r.Z for r in rows]
    fig = Figure(size=(1400, 880), fontsize=16)
    Label(fig[1, 1:2], "Surface phytoplankton and zooplankton", fontsize=23)

    ax1 = Axis(fig[2, 1], title="Global area-weighted biomass",
               xlabel="Model day", ylabel=L"Biomass ($\mathrm{mmol\,N\,m^{-3}}$)",
               yscale=log10, yautolimitmargin=(0.08, 0.22))
    lines!(ax1, x, positive(p); color=COLORS[1], label="Phytoplankton P", linewidth=2.4)
    lines!(ax1, x, positive(z); color=COLORS[2], label="Zooplankton Z", linewidth=2.4)
    no_positive_message!(ax1, vcat(p, z))
    decade_ticks!(ax1, vcat(p, z))
    inside_legend(ax1)

    ax2 = Axis(fig[2, 2], title="Zooplankton relative to phytoplankton",
               xlabel="Model day", ylabel="Mean Z / mean P", yscale=log10,
               yautolimitmargin=(0.08, 0.25))
    ax4 = Axis(fig[3, 2], title="Regional zooplankton",
               xlabel="Model day", ylabel=L"$Z$ ($\mathrm{mmol\,N\,m^{-3}}$)", yscale=log10,
               yautolimitmargin=(0.08, 0.25))
    all_ratios, all_z = Float64[], Float64[]
    for (i, region) in enumerate(regions)
        regional = sort([r for r in diagnostics.series if r.region == region]; by=r -> r.day)
        days = [r.day for r in regional]
        ratios, zs = [r.ratio for r in regional], [r.Z for r in regional]
        color = COLORS[mod1(i, length(COLORS))]
        lines!(ax2, days, positive(ratios); color, label=region, linewidth=2)
        lines!(ax4, days, positive(zs); color, label=region, linewidth=2)
        append!(all_ratios, ratios)
        append!(all_z, zs)
    end
    no_positive_message!(ax2, all_ratios)
    no_positive_message!(ax4, all_z)
    decade_ticks!(ax2, all_ratios)
    decade_ticks!(ax4, all_z)
    inside_legend(ax2; position=:rt)
    inside_legend(ax4)

    ax3 = Axis(fig[3, 1], title="Biomass relative to initial value",
               xlabel="Model day", ylabel="Biomass / initial positive biomass",
               yscale=log10, yautolimitmargin=(0.08, 0.22))
    np, nz = normalized(p), normalized(z)
    lines!(ax3, x, np; color=COLORS[1], label="Phytoplankton P", linewidth=2.4)
    lines!(ax3, x, nz; color=COLORS[2], label="Zooplankton Z", linewidth=2.4)
    hlines!(ax3, [1.0]; color=:gray55, linestyle=:dash, linewidth=1)
    no_positive_message!(ax3, vcat(np, nz))
    decade_ticks!(ax3, vcat(np, nz))
    inside_legend(ax3)

    linkxaxes!(ax1, ax2, ax3, ax4)
    colgap!(fig.layout, 32)
    rowgap!(fig.layout, 22)
    return fig
end

# Estimated biological gains and losses of Z.
function grazing_series(diagnostics)
    isnothing(diagnostics.parameters) &&
        error("Grazing parameters are unavailable; inspect the biomass figures first.")
    rows = global_rows(diagnostics)
    x = [r.day for r in rows]
    fig = Figure(size=(1180, 1040), fontsize=16)
    Label(fig[1, 1], "Surface grazing and zooplankton budget: snapshot estimates", fontsize=22,
          tellwidth=false)
    ax1 = Axis(fig[2, 1], title="Global grazing fluxes",
               xlabel="Model day", ylabel=L"\mathrm{mmol\,N\,m^{-3}\,day^{-1}}",
               yautolimitmargin=(0.05, 0.25))
    lines!(ax1, x, finite([r.gp for r in rows]); color=COLORS[1], linewidth=2.3,
           label="Grazing on phytoplankton")
    lines!(ax1, x, finite([r.gtot for r in rows]); color=COLORS[2], linewidth=2.3,
           label="Total grazing")
    inside_legend(ax1)

    ax2 = Axis(fig[3, 1], title="Global zooplankton gains and losses",
               xlabel="Model day", ylabel=L"\mathrm{mmol\,N\,m^{-3}\,day^{-1}}",
               yautolimitmargin=(0.05, 0.25))
    for (field, label, color) in ((:assimilation, "Assimilation", COLORS[3]),
                                 (:z_excretion, "Excretion", COLORS[1]),
                                 (:z_mortality, "Mortality", COLORS[2]))
        lines!(ax2, x, finite([getproperty(r, field) for r in rows]);
               color, label, linewidth=2.3)
    end
    inside_legend(ax2)

    ax3 = Axis(fig[4, 1], title="Potential net biological Z tendency",
               xlabel="Model day", ylabel=L"\mathrm{mmol\,N\,m^{-3}\,day^{-1}}",
               yautolimitmargin=(0.1, 0.25))
    hlines!(ax3, [0.0]; color=:gray50, linestyle=:dash, linewidth=1)
    lines!(ax3, x, finite([r.z_bio_tendency for r in rows]);
           color=COLORS[4], linewidth=2.3,
           label="Assimilation − excretion − mortality")
    inside_legend(ax3)
    linkxaxes!(ax1, ax2, ax3)
    colsize!(fig.layout, 1, Relative(1))
    rowgap!(fig.layout, 22)
    return fig
end

function log_ticks(limits)
    all(isfinite, limits) && 0 < limits[1] < limits[2] ||
        error("Logarithmic color limits must be finite, positive, and increasing.")
    exponents = collect(ceil(log10(limits[1])):floor(log10(limits[2])))
    isempty(exponents) && (exponents = collect(log10.(limits)))
    return (exponents, [@sprintf("%g", 10.0^e) for e in exponents])
end

function display_log_field(values, diagnostics)
    map, wet = diagnostics.mapping.map, diagnostics.catalog.wet
    length(values) == length(wet) || error("Monthly field and native wet-mask lengths differ.")
    result = fill(NaN, size(map))
    for i in eachindex(map)
        source = map[i]
        1 <= source <= length(values) || continue
        wet[source] || continue
        value = values[source]
        if isfinite(value) && value > 0
            result[i] = log10(value)
        end
    end
    return result
end

month_start(value::Date) = Date(year(value), month(value), 1)
month_start(value::DateTime) = month_start(Date(value))
month_start(value::AbstractString) = month_start(Date(length(value) == 7 ? value * "-01" : value))

# Monthly P, Z and Z/P maps.
function monthly_maps(diagnostics; months=nothing, p_limits=(1e-3, 3.0),
                      z_limits=(1e-5, 1.0), ratio_limits=(1e-3, 3.0))
    ordered = sort(collect(diagnostics.monthly); by=r -> r.month)
    isempty(ordered) && error("No complete monthly means are available to plot.")
    if isnothing(months)
        ordered = ordered[1:min(4, length(ordered))]
    else
        requested = unique(month_start.(collect(months)))
        available = Set(r.month for r in ordered)
        missing = [m for m in requested if !(m in available)]
        isempty(missing) || error("Requested months are unavailable: $(join(string.(missing), ", ")).")
        ordered = [r for r in ordered if r.month in requested]
        isempty(ordered) && error("No months were selected.")
    end
    limits = (p_limits, z_limits, ratio_limits)
    ticks = log_ticks.(limits)
    nmonths = length(ordered)
    mapping = diagnostics.mapping
    size(mapping.map) == (length(mapping.lon), length(mapping.lat)) ||
        error("Display map shape must match (length(lon), length(lat)).")
    fig = Figure(size=(1450, 280nmonths + 150), fontsize=15)
    Label(fig[1, 2:4], "Surface phytoplankton and zooplankton monthly means", fontsize=23)
    titles = ("Phytoplankton P", "Zooplankton Z", "Z / P")
    barlabels = (L"$P$ ($\mathrm{mmol\,N\,m^{-3}}$, log scale)", L"$Z$ ($\mathrm{mmol\,N\,m^{-3}}$, log scale)",
                 "Z / P (log scale)")
    for (m, result) in enumerate(ordered)
        length(result.P) == length(result.Z) || error("Monthly P and Z lengths differ.")
        ratio_floor = hasproperty(diagnostics, :ratio_floor) ? diagnostics.ratio_floor : 1e-8
        ratio = [isfinite(p) && p > ratio_floor && isfinite(z) && z > 0 ? z / p : NaN
                 for (p, z) in zip(result.P, result.Z)]
        row = m + 1
        Label(fig[row, 1], Dates.format(result.month, dateformat"yyyy-mm");
              fontsize=16, font=:bold, tellheight=false)
        for (i, values) in enumerate((result.P, result.Z, ratio))
            column = i + 1
            ax = Axis(fig[row, column], title=m == 1 ? titles[i] : "",
                      xlabel="Longitude", ylabel=i == 1 ? "Latitude" : "",
                      aspect=DataAspect(), backgroundcolor=:gray90,
                      xticks=-180:90:180, yticks=-90:45:90)
            limits!(ax, -180, 180, -90, 90)
            m < nmonths && hidexdecorations!(ax; grid=false)
            i > 1 && hideydecorations!(ax; grid=false)
            hm = heatmap!(ax, mapping.lon, mapping.lat, display_log_field(values, diagnostics);
                          nan_color=:transparent, colorrange=log10.(limits[i]),
                          colormap=i == 3 ? :plasma : :viridis)
            if m == nmonths
                Colorbar(fig[nmonths + 2, column], hm; vertical=false,
                         width=Relative(0.78), height=10, halign=:center,
                         ticks=ticks[i], label=barlabels[i], labelsize=13, ticklabelsize=13)
            end
        end
        rowsize!(fig.layout, row, Aspect(2, 0.5))
    end
    colsize!(fig.layout, 1, 85)
    for column in 2:4
        colsize!(fig.layout, column, Auto(1))
    end
    colgap!(fig.layout, 40)
    rowgap!(fig.layout, 18)
    resize_to_layout!(fig)
    return fig
end

end


Main.ZooPlots

## 1. Calculate surface means

The time series use area-weighted means at each output time. Monthly means are calculated from the daily snapshots. For the ratios, I divide mean $Z$ by mean $P$.

The model's chlorophyll conversion is $1.31\ \mathrm{mg\,Chl\,(mmol\,N)^{-1}}$.


In [ ]:
catalog = Base.invokelatest() do
    ZooDiagnostics.inspect_run(segment_files; day0=day0, first_day=first_day,
                               last_day=last_day, grazing=!isnothing(grazing_parameters),
                               segment_names=run_names)
end

diagnostics = Base.invokelatest() do
    ZooDiagnostics.diagnose(catalog; regions=regions, months=map_months,
                            parameters=grazing_parameters, ratio_floor=ratio_floor)
end;

mkpath(figure_dir)
Base.invokelatest() do
    ChlComparison.write_catalog(figure_dir, catalog)
    ZooDiagnostics.write_summary(joinpath(figure_dir, "surface_P_Z_daily_summary.csv"), diagnostics)
end
open(joinpath(figure_dir, "diagnostic_settings.txt"), "w") do io
    println(io, "run_names (oldest first) = ", run_names, "\nday0 = ", day0)
    println(io, "Files (two spatial ranks per segment):\n", join(catalog.paths, '\n'))
    println(io, "Overlap policy = later complete rank pair; no averaging of duplicate times")
    println(io, "Accepted unique snapshots = ", length(catalog.records))
    println(io, "Actual first/last time = ", first(catalog.records).datetime, " / ", last(catalog.records).datetime)
    println(io, "Parameters assumed for snapshot reconstruction = ", grazing_parameters)
    println(io, "Regions = ", regions, "\nRatio floor = ", ratio_floor)
    println(io, "Julia = ", VERSION, "\nProject = ", Base.active_project())
    for dep in values(Pkg.dependencies())
        dep.name in ("OceanBioME", "Oceananigans") && println(io, dep.name, " = ", dep.version)
    end
end
println("Summary and settings saved in ", figure_dir)


ice_glorys01_2_iron: 

## 2. P and Z through time

These plots show the global and regional changes. The relative-change panel divides each series by its first positive value across the combined runs.


In [ ]:
Base.invokelatest() do
    fig = ZooPlots.time_series(diagnostics)
    display(fig)
    save(joinpath(figure_dir, "surface_P_Z_time_series.png"), fig)
end


## 3. Grazing and the Z budget

I estimate grazing from $P$, $Z$ and small detritus (`sPOM`) using the parameters above. These are entered manually and need to match the model run.

$$
\left(\frac{dZ}{dt}\right)_{\mathrm{bio}}
= \mathrm{assimilation} - \mathrm{excretion} - \mathrm{mortality}
$$

Rates are in $\mathrm{mmol\,N\,m^{-3}\,day^{-1}}$. They are estimates at each output time and exclude transport and mixing.


In [ ]:
if !isnothing(grazing_parameters)
    Base.invokelatest() do
        fig = ZooPlots.grazing_series(diagnostics)
        display(fig)
        save(joinpath(figure_dir, "surface_grazing_Z_budget.png"), fig)
        global_rows = filter(r -> r.region == "Global", diagnostics.series)
        println("Valid wet area for reconstructed rates (min/max fraction): ",
                extrema(r.valid_fraction for r in global_rows))
    end
else
    println("Grazing reconstruction disabled; P/Z diagnostics remain available.")
end


## 4. Monthly maps

The maps show $P$, $Z$ and $Z/P$ for complete months, four months per page. Each column uses the same color scale across months.

$P$ and $Z$ are in $\mathrm{mmol\,N\,m^{-3}}$; $Z/P$ has no units. Map colors use a log scale. Incomplete months are skipped here but remain in the time series.


In [ ]:
if !isempty(diagnostics.monthly)
    map_grid = Base.invokelatest() do
        ZooDiagnostics.display_mapping(catalog; spacing=map_spacing_deg, radius=map_radius_deg)
    end
    mapped_diagnostics = merge(diagnostics, (; mapping=map_grid))
    Base.invokelatest() do
        months_per_page >= 1 || error("months_per_page must be positive")
        all_months = sort(getproperty.(diagnostics.monthly, :month))
        for start in 1:months_per_page:length(all_months)
            page = all_months[start:min(start + months_per_page - 1, length(all_months))]
            fig = ZooPlots.monthly_maps(mapped_diagnostics; months=page,
                                       p_limits=(1e-3, 3.0), z_limits=(1e-5, 1.0), ratio_limits=(1e-3, 3.0))
            label = Dates.format(first(page), dateformat"yyyymm") * "_" *
                    Dates.format(last(page), dateformat"yyyymm")
            display(fig)
            save(joinpath(figure_dir, "surface_P_Z_monthly_maps_$(label).png"), fig)
        end
    end
else
    println("No complete requested month in the merged timeline. The time-series figures are still useful.")
end


## What I want to check

Does $Z$ catch up with $P$, and does grazing increase as $Z$ grows? A low $Z/P$ alone does not show that zooplankton is too low.

Figures and summaries are saved in `figure_dir`.
